# ✅ Stage 3 — Data Validation

**Goal:** Detect and report data quality issues before imputation.

This notebook covers:
- **JSON Schema validation** (type checking, required fields, constraints)
- **Missing required / optional field detection**
- **Duplicate flagging**
- **Outlier detection** (extreme times, unusual serving counts)
- **Parse-warning review**
- **Saving `validation_report.csv`**

---


## 0. Setup

In [ ]:
import sys, json, os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import pandas as pd
from src.validation import validate_all, check_outliers

INPUT_FILE   = os.path.join(REPO_ROOT, 'data', 'enriched_data.json')
SCHEMA_FILE  = os.path.join(REPO_ROOT, 'recipe_schema.json')
OUTPUT_REPORT = os.path.join(REPO_ROOT, 'data', 'validation_report.csv')

print('Setup complete.')


In [ ]:
with open(INPUT_FILE,  'r', encoding='utf-8') as f:
    recipes = json.load(f)
with open(SCHEMA_FILE, 'r', encoding='utf-8') as f:
    schema  = json.load(f)

print(f'Loaded {len(recipes)} enriched recipes | Schema: {SCHEMA_FILE}')


## 1. JSON Schema Overview

In [ ]:
print(json.dumps(schema, indent=2))


## 2. Run Full Validation

In [ ]:
report_rows = validate_all(recipes, schema)
report_df   = pd.DataFrame(report_rows)

print(f'Total issues found: {len(report_df)}')
if not report_df.empty:
    print(report_df['severity'].value_counts().to_string())


## 3. Issues by Type

In [ ]:
if not report_df.empty:
    print('Issues by check_type:')
    print(report_df['check_type'].value_counts().to_string())


## 4. Errors (must fix)

In [ ]:
if not report_df.empty:
    errors = report_df[report_df['severity'] == 'error']
    print(f'{len(errors)} error(s):')
    if not errors.empty:
        display(errors[['recipe_id','recipe_name','check_type','field','description']])
    else:
        print('  ✓ No errors found!')


## 5. Warnings (will be imputed or reviewed)

In [ ]:
if not report_df.empty:
    warnings = report_df[report_df['severity'] == 'warning']
    print(f'{len(warnings)} warning(s):')
    display(warnings[['recipe_id','recipe_name','check_type','field','description']])


## 6. Missing Value Heatmap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fields = ['name', 'ingredients', 'instructions', 'prep_time_minutes', 'servings']

def is_missing(r, f):
    v = r.get(f)
    return 1 if (v is None or v == [] or v == '') else 0

matrix = [[is_missing(r, f) for f in fields] for r in recipes]
matrix_np = np.array(matrix)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(matrix_np, cmap='RdYlGn_r', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(fields)))
ax.set_xticklabels(fields, rotation=30, ha='right')
ax.set_yticks(range(len(recipes)))
ax.set_yticklabels([r['recipe_id'] for r in recipes])
ax.set_title('Missing Value Heatmap  (red = missing, green = present)', fontsize=13)
plt.colorbar(im, ax=ax, ticks=[0, 1], shrink=0.7)
plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'data', 'missing_heatmap.png'), dpi=120)
plt.show()
print('Missing value rates:')
for i, f in enumerate(fields):
    pct = matrix_np[:, i].mean() * 100
    print(f'  {f:<25} {pct:.1f}% missing')


## 7. Duplicate Summary

In [ ]:
dup_rows = [r for r in recipes if r.get('_meta', {}).get('is_duplicate')]
print(f'Duplicates found: {len(dup_rows)}')
for d in dup_rows:
    print(f"  {d['recipe_id']} '{d['name']}' → duplicates {d['_meta']['duplicate_of']}")


## 8. Outlier Check

In [ ]:
print('Outlier check per recipe:')
any_found = False
for r in recipes:
    issues = check_outliers(r)
    if issues:
        any_found = True
        for issue in issues:
            print(f"  ⚠ [{r['recipe_id']}] {issue}")
if not any_found:
    print('  ✓ No outliers detected.')


## 9. Save Validation Report

In [ ]:
if not report_df.empty:
    report_df.to_csv(OUTPUT_REPORT, index=False, encoding='utf-8')
    print(f'✅ Saved {len(report_df)} rows → {OUTPUT_REPORT}')
else:
    open(OUTPUT_REPORT, 'w').close()
    print('✅ No issues — empty report saved.')
print('   Next step: run 04_imputation.ipynb')
